# Profil de demande journalière

Ce notebook montre comment construire et lire le profil de demande moyen sur une journée type : nombre moyen de personnes en déplacement par tranche horaire, ventilé par mode et par phase.

Fonctions couvertes : `build_daily_demand_profile()`, `compute_mobility_indicators()`, `plot_indicator_bars()`.

## 1. Préparer l'environnement

In [ ]:
import pandas as pd

import xyt_gps as xyt

## 2. Générer les données et calculer les indicateurs

Le profil de demande s'appuie sur les legs (pour les plages horaires) et sur la table `person_day` (pour le dénominateur : nombre de jours actifs par phase). Les deux sont disponibles après `run_mobility_pipeline()`.

In [ ]:
phases = (
    xyt.Phase("Phase1", "2026-04-01", "2026-04-14"),
    xyt.Phase("Phase2", "2026-04-15", "2026-04-28"),
)

config = xyt.ProjectConfig(
    experiment_name="demo-demande",
    raw_data_dir=".",
    phases=phases,
)

synthetic = xyt.generate_synthetic_declic_gps(
    experiments=(xyt.SyntheticExperiment("demo-demande", phases=phases),),
    users_per_experiment=20,
    random_state=42,
    validate=True,
)

raw, dataset, indicators = xyt.run_mobility_pipeline(
    config,
    raw=synthetic.raw,
    resample_missing_days=True,
    mode_col="mode_niv1",
    include_zero_days=True,
)

print(f"legs : {len(dataset.legs):,} lignes · {dataset.legs['user_id'].nunique()} utilisateurs")

## 3. Construire le profil de demande

`build_daily_demand_profile()` découpe les legs en tranches de 5 minutes (défaut) et calcule le nombre moyen de personnes en déplacement à chaque tranche, normalisé par le nombre de jours actifs de la phase. Le résultat est un dictionnaire structuré par phase et par mode.

In [ ]:
profile = xyt.build_daily_demand_profile(
    dataset.legs,
    indicators.person_day,
    mode_col="mode_niv1",
)

print(f"résolution : {profile['bin_minutes']} min · {len(profile['time_labels'])} tranches")
print(f"phases : {[p['phase'] for p in profile['phases']]}")
print(f"modes disponibles : {[s['mode'] for s in profile['phases'][0]['series']]}")

## 4. Lire le profil sous forme tabulaire

Chaque phase du dictionnaire contient une liste de séries temporelles (une par mode). On les convertit en DataFrame pour une lecture directe : les colonnes sont les modes, les lignes sont les tranches horaires de la journée.

In [ ]:
def profile_to_dataframe(profile, phase_name=None):
    """Convertit une phase du profil de demande en DataFrame horaire."""
    phase_data = next(
        (p for p in profile["phases"] if phase_name is None or p["phase"] == phase_name),
        profile["phases"][0],
    )
    labels = profile["time_labels"]
    keep_every = 60 // profile["bin_minutes"]  # une étiquette par heure
    display_labels = [lbl if i % keep_every == 0 else "" for i, lbl in enumerate(labels)]
    df = pd.DataFrame(
        {s["mode"]: s["values"] for s in phase_data["series"]},
        index=display_labels,
    )
    df.index.name = "heure"
    return df, phase_data["phase"], phase_data["n_days"]


df_phase1, phase_name, n_days = profile_to_dataframe(profile, "Phase1")
print(f"{phase_name} · {n_days} jours actifs")
df_phase1.head(24)

In [ ]:
# Heatmap horaire : une ligne par heure (agréger les tranches), une colonne par mode
bin_min = profile["bin_minutes"]
per_hour = 60 // bin_min

df_hourly = (
    df_phase1
    .assign(_hour=lambda d: [f"{i // per_hour:02d}h" for i in range(len(d))])
    .groupby("_hour")
    .mean()
)

(
    df_hourly
    .style
    .background_gradient(cmap="Purples", axis=None, vmin=0)
    .format("{:.2f}")
    .set_caption(f"{phase_name} · personnes moyennes en déplacement par heure et par mode")
)

## 5. Comparer les phases

La même lecture peut être faite pour la deuxième phase et comparée ligne à ligne.

In [ ]:
frames = {}
for phase_profile in profile["phases"]:
    df, pname, n_days = profile_to_dataframe(profile, phase_profile["phase"])
    df_h = (
        df
        .assign(_hour=lambda d: [f"{i // per_hour:02d}h" for i in range(len(d))])
        .groupby("_hour")
        .mean()
    )
    # garder uniquement la colonne tous modes
    frames[pname] = df_h["Tous modes"] if "Tous modes" in df_h.columns else df_h.iloc[:, 0]

(
    pd.concat(frames, axis=1)
    .style
    .background_gradient(cmap="YlOrRd", axis=None, vmin=0)
    .format("{:.2f}")
    .set_caption("Tous modes · personnes moyennes en déplacement par heure, comparaison des phases")
)

## 6. Profil intégré dans la carte d'identité des indicateurs

`compute_mobility_indicators()` stocke automatiquement le profil dans `indicators.metadata`. `plot_indicator_bars()` l'affiche directement en bas de la carte d'identité sans étape supplémentaire.

In [ ]:
xyt.plot_indicator_bars(
    indicators,
    table="population",
    metrics=["distance_km_per_day", "trips_per_day"],
    group_col="mode",
    facet_col="phase",
    title="Indicateurs avec profil de demande",
    show_demand_profile=True,
)

## À retenir

`build_daily_demand_profile()` produit un dictionnaire structuré, pas une figure. Cette séparation permet de :

- stocker le profil avec les indicateurs dans `IndicatorResult.metadata` ;
- l'afficher via `plot_indicator_bars(show_demand_profile=True)` dans un rapport HTML ;
- le convertir en DataFrame pour toute analyse ad hoc (pic du matin, comparaison de phases, export CSV).

| Paramètre | Défaut | Usage |
|---|---|---|
| `bin_minutes` | 5 | résolution temporelle des courbes |
| `heatmap_bin_minutes` | 60 | résolution de la heatmap motifs |
| `mode_col` | `mode_niv1` | ventilation modale |
| `motif_col` | auto-inféré | heatmap motif/motif si `purpose_niv1` présent |